In [ ]:
import torch
import mmap
import random
import pickle

import argparse
parser = argparse.ArgumentParser(description='Train a GPT model.')
parser.add_argument("-bs", type=int, required=True, help="select batch size")
args = parser.parse_args()
print(f"Batch size selected: {args.bs}")
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

block_size = args.bs
batch_size = 128
max_iters = 3000
learning_rate = 3e-4
eval_iter = 250
n_embd = 384
n_layer = 1
n_head = 1
dropout = 0.2

cuda


In [2]:
chars = ""
with open ('Mdm_Bovary.txt', 'r', encoding='utf-8') as f:
    text = f.read()
    chars = sorted(list(set(text)))

vocab_size = len(chars)


In [3]:
string_to_int = { ch:i for i,ch in enumerate (chars) }
int_to_string = { i:ch for i,ch in enumerate (chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join(int_to_string[i] for i in l)

In [4]:


def get_random_chunk(split):
    filename = 'train_data.txt' if split == 'train' else 'val_data.txt'
    with open(filename, 'rb') as f:
        with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:
            file_size = len(mm)
            start_pos = random.randint(0, (file_size) - block_size * batch_size)
            mm.seek(start_pos)
            block = mm.read(block_size * batch_size-1)
            decoded_block = block.decode('utf-8', errors='ignore').replace('\r', ' ')
            data = torch.tensor(encode(text), dtype=torch.long)
    return data

def get_batch(split): #decide whether to use train or val data
    data = get_random_chunk(split)
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y


In [5]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iter)
        for k in range(eval_iter):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
class Head(nn.Module):
    """one head of self attention"""

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x) # (B,T,C)
        q = self.query(x) # (B,T,C)

        #compute attention scores
        wei = q @ k.transpose(-2,-1) * C**-0.5 #(B,T,C) @ (B,C,T) -> (B,T,T)
        wei = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf')) #mask out future positions
        wei = F.softmax(wei, dim=-1) #(B,T,T)
        wei = self.dropout(wei)

        v = self.value(x) #(B,T,C)
        out = wei @ v #(B,T,T) @ (B,T,C) -> (B,T,C)
        return out
 
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        y = self.sa(x) #self attention
        x = self.ln1(x + y) #add and norm
        y = self.ffwd(x) #feed forward
        x = self.ln2(x + y) #add and norm
        return x


class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) #final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights) #apply weight initialization with std = 0.02

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        B, T = index.shape

        tok_emb = self.token_embedding_table(index) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, index, max_new_tokens):
        #index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            index_cond = index[:, -block_size:] #crop context to block size
            #get predictions
            logits, loss = self.forward(index_cond)

            #focus only on last time step
            logits = logits[:, -1, :] #becomes(B,C)
            probs = F.softmax(logits, dim=-1) # (B,C)

            #sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B,1)

            #append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B,T+1)

        return index

model = GPTLanguageModel(vocab_size)
print('loading model parameters from model-01.pkl...')
with open('model-01.pkl', 'rb') as f:
    model = pickle.load(f)
print('model parameters loaded successfully.')
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)
    
    


4î0Ma(ÀÊçÇi(»_W:à*Pç ô’œêa»ÎBî575G0LÀ;;«)nÎ-Nâ0ü:9f;1D7ôCWdBÈr(xB!ô
SÈûWhÈNî)ÉSÊn« s(1è(ÎœoU’Îc’kQ888ÈskQEÉecQVÀûû8DuOLÈkS
-œcÇ0ëbR
gXàNÀi’XNÈVy»7ohîlQÊx7T

éDÉThcw4àéëhsïwGïîîXÉ*?ü_sF»çnxùOIqg!ÉtaoD5À:ST.4bH9îN61.jelG éÎœ«Nd—adl)64Fàûw87e5EX7ml—3)T)
C4jùt46lSyiUœô0U9îg49VA4(e?h_kzQ—I6bmG5ê-ô:»QmPS»lé3-È;_Ê°ec1»,p3lQJLTdH(uÀUêR:60è2qlÀ9wô(nI«j*xÊlx_ JÎ(lTWbbà7àsbBAÉSÊ«qr!’QGcïtçDU.5Q2Oky4l,WV0ÈU(ùx3ph»b4(Mù7.â’O;3»ïl Î81Arnh3mAOûGGh(Î0«Î35GEzEBoA2b:tAMQ»I-qbbY*î*zAThd2üc14wniG(sqOgE4É,T«PIW-ÎÈa



In [ ]:

#optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

for iter in range(max_iters):
    if iter % eval_iter == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.4f}, val loss: {losses['val']:.4f}")


    #sample a batch of data
    xb, yb = get_batch('train')

    #evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

with open('model-01.pkl', 'wb') as f:
    pickle.dump(model, f)
print("Model saved to model-01.pkl")


step: 0, train loss: 4.6913, val loss: 4.6914
step: 250, train loss: 1.8970, val loss: 1.8970
step: 500, train loss: 1.5898, val loss: 1.5917
step: 750, train loss: 1.4470, val loss: 1.4452
step: 1000, train loss: 1.3442, val loss: 1.3452
step: 1250, train loss: 1.2818, val loss: 1.2831
step: 1500, train loss: 1.2316, val loss: 1.2328
step: 1750, train loss: 1.1931, val loss: 1.1933
step: 2000, train loss: 1.1578, val loss: 1.1610
step: 2250, train loss: 1.1275, val loss: 1.1288
step: 2500, train loss: 1.0994, val loss: 1.1005
step: 2750, train loss: 1.0706, val loss: 1.0717
1.1214075088500977
